# AksaraLine 2/2 — train the recognizers

GPU. Consumes the corpus kernel's output, so no rendering happens here.

`06_run_matrix.py` skips cells that already have a `result.json`, so if the
session is cut short, re-running with the previous output attached resumes
rather than starting over.


In [ ]:
import os, sys, subprocess, zipfile, time
from pathlib import Path

BRANCH  = 'aksara-seq'
REPO    = Path('/kaggle/working/aksara_OCR')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/phoenixfin/aksantara-ocr.git',
                    str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'aksara_seq' / 'src'))
print('commit:', subprocess.run(['git','rev-parse','--short','HEAD'],
      capture_output=True, text=True).stdout.strip())


In [ ]:
import zipfile
INPUT  = Path('/kaggle/input')
CORPUS = Path('/kaggle/working/corpus/v1')
RECOG  = Path('/kaggle/working/build/recog')
SCRIPTS = ['Sunda', 'Jawa', 'Bali', 'Lontara']

# The corpus kernel emits one archive per script plus dataset_meta.json.
# Find them by shape rather than by mount path: datasets land under
# /kaggle/input/datasets/..., kernel outputs under /kaggle/input/notebooks/...,
# and the depth differs between the two.
def find_files(root, pattern, max_depth=10):
    hits, frontier = [], [root]
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                for q in d.iterdir():
                    if q.is_dir():
                        if q.name != 'images':
                            nxt.append(q)
                    elif pattern(q):
                        hits.append(q)
            except OSError:
                pass
        if hits or not nxt:
            break
        frontier = nxt
    return hits

archives = find_files(INPUT, lambda q: q.name.startswith('corpus_')
                      and q.suffix in ('.zip', '.bin'))
meta = find_files(INPUT, lambda q: q.name == 'dataset_meta.json')
assert archives, ('no corpus_<script>.zip found under /kaggle/input; '
                  'attach the aksaraline-corpus kernel output')
print('archives:', [a.name for a in archives])

CORPUS.mkdir(parents=True, exist_ok=True)
for a in archives:
    import time as _t
    t0 = _t.time()
    with zipfile.ZipFile(a) as z:
        z.extractall(CORPUS)
    print(f'{a.name}: extracted in {_t.time()-t0:.0f}s')
if meta:
    import shutil as _sh
    _sh.copy2(meta[0], CORPUS / 'dataset_meta.json')

for s in SCRIPTS:
    n = sum(1 for _ in (CORPUS / s / 'train' / 'images').glob('*.png'))
    print(f'  {s:9s} {n} train lines')
    assert n == 20000, f'{s} has {n} train lines, expected 20000'

import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    raise RuntimeError('no GPU allocated; enable the accelerator')

cap = torch.cuda.get_device_capability(0)
print('compute capability: sm_%d%d' % cap)
print('torch build supports:', torch.cuda.get_arch_list())
try:
    (torch.zeros(8, 8, device='cuda') @ torch.zeros(8, 8, device='cuda')).cpu()
    print('GPU kernel launch OK')
except Exception as exc:
    raise RuntimeError(
        'GPU present but unusable: ' + str(exc) + '. This happens on a P100 '
        '(sm_60), which the PyTorch build on Kaggle no longer targets. Set '
        '"machine_shape": "NvidiaTeslaT4" in kernel-metadata.json, or pick T4 '
        'in the notebook settings, and re-run.') from exc


## Detection baseline (YOLO)

Not a competitor to CTC so much as a measurement of how *segmentable* this
corpus is. Glyphs here never overlap by construction, so detection is
easier here than on real cursive hands -- if a small detector does well,
that bounds what the corpus can claim about real manuscripts, which is a
useful number to report rather than hide.

One script (Bali, 127 classes) and an untuned yolo11n at 640px, so read it
as a floor on detectability, not as a tuned result.


In [ ]:
!pip install -q ultralytics 2>&1 | tail -2


In [ ]:
!python aksara_seq/scripts/05_export_yolo.py --corpus {CORPUS} --script Bali --classes syllable --out /kaggle/working/yolo_Bali


In [ ]:
!yolo detect train data=/kaggle/working/yolo_Bali/data.yaml model=yolo11n.pt imgsz=640 rect=True epochs=20 batch=16 project=/kaggle/working/build/yolo name=Bali workers=2 seed=0


In [ ]:
!yolo detect val data=/kaggle/working/yolo_Bali/data.yaml model=/kaggle/working/build/yolo/Bali/weights/best.pt split=test imgsz=640 rect=True project=/kaggle/working/build/yolo name=Bali_test
